# Lab 03 – Production Rollout & Incident Playbook

**Scenario:** You will execute a dry run of the production rollout plan, including go/no-go checkpointing, incident response rehearsal, and post-launch communications. Outputs support the CAB approval and launch readiness review.

## Objectives
- Finalize production rollout checklist and go/no-go criteria
- Simulate deployment, canary, and rollback sequences
- Execute incident tabletop and document runbook updates
- Prepare stakeholder communication packets and status updates

## Step 1 – Rollout Timeline
Build the day-of-launch plan with timestamps, owners, and checkpoints.

In [ ]:
from datetime import datetime, timedelta
from tabulate import tabulate

start = datetime.strptime('2025-12-19 13:00', '%Y-%m-%d %H:%M')
schedule = [
    ('T-60m', start - timedelta(minutes=60), 'Finalize go/no-go, verify dashboards', 'Release Manager'),
    ('T-30m', start - timedelta(minutes=30), 'Enable canary flag, monitor metrics', 'SRE Lead'),
    ('T-15m', start - timedelta(minutes=15), 'Stakeholder comms prep, status page draft', 'Comms Lead'),
    ('T+0m', start, 'Begin progressive rollout (25% traffic)', 'DevOps'),
    ('T+30m', start + timedelta(minutes=30), 'Increase to 100% if metrics stable', 'Incident Commander'),
    ('T+60m', start + timedelta(minutes=60), 'Send launch announcement + support brief', 'Product Lead'),
]
table = tabulate(schedule, headers=['Marker', 'Timestamp', 'Action', 'Owner'], tablefmt='github')
print(table)
Path('artifacts').mkdir(exist_ok=True)
Path('artifacts/launch-timeline.md').write_text(table)
print('Saved timeline to artifacts/launch-timeline.md')

## Step 2 – Go/No-Go Checklist
Evaluate readiness across technical, compliance, support, and stakeholder dimensions. Document evidence links.

In [ ]:
import csv

scorecard = Path('resources/production-slo-scorecard.csv')
if not scorecard.exists():
    raise FileNotFoundError('Populate production-slo-scorecard.csv before running this step.')

gates = [
    ('Technical readiness', 'All SLO dashboards green'),
    ('Compliance readiness', 'Audit artifacts uploaded'),
    ('Support readiness', 'Support scripts approved'),
    ('Stakeholder readiness', 'Exec sponsor sign-off recorded'),
]
Path('artifacts/go-no-go-checklist.csv').write_text('category,notes
' + '
'.join([f
 for g in gates]))
print('Go/no-go checklist exported to artifacts/go-no-go-checklist.csv')

## Step 3 – Deployment Dry Run
Simulate the deployment pipeline, canary rollout, and rollback procedure. Log outputs for CAB.

In [ ]:
from pipelines.release import ReleaseOrchestrator

orchestrator = ReleaseOrchestrator(environment='production')

plan = orchestrator.plan_release(release_id='rel-placeholder')
print('Release plan summary:')
print(plan.summary())

canary_result = orchestrator.execute_canary(traffic_percent=25)
print(f'Canary outcome: {canary_result.status}')

if not canary_result.passed:
    rollback = orchestrator.rollback(release_id='rel-placeholder')
    print(f'Rollback executed: {rollback.status}')
else:
    orchestrator.advance_to_full_release()
    print('Rollout advanced to 100%. Monitor SLOs for 60 minutes.')

orchestrator.capture_evidence(destination='artifacts/release-evidence/')
print('Release evidence stored in artifacts/release-evidence/')

## Step 4 – Incident Tabletop
Run the incident scenario provided by the facilitator. Document decisions, comms, and follow-ups.

In [ ]:
from incident.simulation import TabletopRecorder

recorder = TabletopRecorder(runbook_path='resources/incident-playbook-template.md')
scenario = recorder.load_scenario('llm-provider-outage')
recorder.start(scenario=scenario)
recorder.log_event('LLM provider returns 5xx. Trigger fallback provider.')
recorder.log_decision(owner='Incident Commander', decision='Shift 100% traffic to backup model, update status page.')
recorder.log_decision(owner='Comms Lead', decision='Send stakeholder alert with ETA 45 minutes.')
recorder.complete()
recorder.export('artifacts/tabletop-report.md')
print('Tabletop report saved to artifacts/tabletop-report.md')

## Step 5 – Communication Packet
Prepare launch announcement, support briefing, and potential rollback message using the communication plan template.

In [ ]:
from communications.templates import build_launch_digest

digest = build_launch_digest(
    audience='executive',
    highlights={
        'value': 'Launching production-ready GenAI assistant with measurable risk reduction',
        'metrics': ['p95 latency 1.8s', 'guardrail block rate 97.5%', 'drift score < 0.2'],
        'call_to_action': 'Share feedback within first 48 hours to prioritize enhancements',
    },
    incident_contact='oncall-genai@enterprise.com',
)
Path('artifacts/launch-communication-exec.html').write_text(digest)
print('Launch communication draft stored at artifacts/launch-communication-exec.html')

## Step 6 – Post-Launch Operating Rhythm
Update the MLOps operating model document with cadences, owners, and reporting templates.

In [ ]:
doc = Path('resources/mlops-operating-model.md')
if not doc.exists():
    raise FileNotFoundError('Populate mlops-operating-model.md prior to the lab.')

with doc.open('a') as fp:
    fp.write('

## Week 9 Launch Cadence Updates
')
    fp.write('- Daily error budget review at 09:00 with SRE lead\n')
    fp.write('- Weekly drift + eval sync every Tuesday with applied science\n')
    fp.write('- Bi-weekly stakeholder adoption review on Thursdays\n')
    fp.write('- Monthly compliance audit touchpoint with legal\n')
print('Appended launch cadence updates to resources/mlops-operating-model.md')

## Submission Checklist
- Launch timeline and go/no-go checklist committed
- Release evidence bundle populated (`artifacts/release-evidence/`)
- Tabletop report attached
- Communication drafts saved for exec/support audiences
- Operating model updated with cadence and ownership details
- Readiness scorecard reflects launch status and evidence links